In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import time
import json
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

DATASET_DIR = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 10
SEED = 42

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass


def collect_paths(folder):
    paths, labels = [], []

    for label, class_name in enumerate(CLASS_NAMES):
        class_folder = os.path.join(folder, class_name)

        if not os.path.isdir(class_folder):
            raise FileNotFoundError(f"Missing folder: {class_folder}")

        for root, _, filenames in os.walk(class_folder):
            for filename in sorted(filenames):
                if filename.lower().endswith(VALID_EXTENSIONS):
                    paths.append(os.path.join(root, filename))
                    labels.append(label)

    return np.asarray(paths), np.asarray(labels, dtype=np.int32)


def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    return image, label


def make_dataset(paths, labels, training, mode):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if mode == "serial":
        ds = ds.map(load_image, num_parallel_calls=1)
    else:
        ds = ds.map(
            load_image,
            num_parallel_calls=tf.data.AUTOTUNE
        )

    if training:
        ds = ds.shuffle(
            min(len(paths), 2000),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE)

    if mode == "parallel":
        ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),

        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(64, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(128, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(4, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


def run_experiment(mode):

    tf.keras.utils.set_random_seed(SEED)

    training_dir = os.path.join(DATASET_DIR, "Training")
    testing_dir = os.path.join(DATASET_DIR, "Testing")

    train_paths, train_labels = collect_paths(training_dir)
    test_paths, test_labels = collect_paths(testing_dir)

    train_paths, val_paths, train_labels, val_labels = train_test_split(
        train_paths,
        train_labels,
        test_size=0.20,
        random_state=SEED,
        stratify=train_labels
    )

    train_ds = make_dataset(train_paths, train_labels, True, mode)
    val_ds = make_dataset(val_paths, val_labels, False, mode)
    test_ds = make_dataset(test_paths, test_labels, False, mode)

    strategy = tf.distribute.OneDeviceStrategy("/CPU:0")

    print("\nExecution Mode: SERIAL")
    print("Device: CPU")
    print("Training Images:", len(train_paths))
    print("Validation Images:", len(val_paths))
    print("Testing Images:", len(test_paths))

    with strategy.scope():
        model = build_model()

    start = time.perf_counter()

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=1
    )

    training_seconds = time.perf_counter() - start

    start = time.perf_counter()

    probabilities = model.predict(test_ds, verbose=0)

    prediction_seconds = time.perf_counter() - start

    predictions = np.argmax(probabilities, axis=1)

    report = classification_report(
        test_labels,
        predictions,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )

    matrix = confusion_matrix(
        test_labels,
        predictions,
        labels=[0, 1, 2, 3]
    )

    print("\nSERIAL CLASSIFICATION REPORT")
    print(report)

    print("SERIAL CONFUSION MATRIX")
    print(matrix)

    total_seconds = training_seconds + prediction_seconds

    print(f"\nTraining Time: {training_seconds:.2f} seconds")
    print(f"Prediction Time: {prediction_seconds:.2f} seconds")
    print(f"Total Time: {total_seconds:.2f} seconds")

    results = {
        "mode": mode,
        "training_seconds": training_seconds,
        "prediction_seconds": prediction_seconds,
        "total_seconds": total_seconds,
        "classification_report": classification_report(
            test_labels,
            predictions,
            labels=[0, 1, 2, 3],
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0
        ),
        "confusion_matrix": matrix.tolist()
    }

    return results


serial_results = run_experiment("serial")

with open("/kaggle/working/serial_results.json", "w") as f:
    json.dump(serial_results, f, indent=2)


I0000 00:00:1794895943.267034      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1794895943.271733      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



Execution Mode: SERIAL
Device: CPU
Training Images: 4480
Validation Images: 1120
Testing Images: 1600
Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 175s 1s/step - accuracy: 0.6705 - loss: 0.7722 - val_accuracy: 0.8339 - val_loss: 0.4912
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 138s 959ms/step - accuracy: 0.8098 - loss: 0.4786 - val_accuracy: 0.8339 - val_loss: 0.4144
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 136s 945ms/step - accuracy: 0.8520 - loss: 0.3842 - val_accuracy: 0.8741 - val_loss: 0.3129
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 138s 956ms/step - accuracy: 0.8942 - loss: 0.2895 - val_accuracy: 0.8929 - val_loss: 0.2854
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 140s 972ms/step - accuracy: 0.9201 - loss: 0.2191 - val_accuracy: 0.9054 - val_loss: 0.2622
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 140s 970ms/step - accuracy: 0.9339 - loss: 0.1733 - val_accuracy: 0.8884 - val_loss: 0.2910
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 137s 948ms/step - accuracy: 0.9592 - loss: 0.1281 - val_accuracy: 0.

In [22]:

tf.keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")

if len(gpus) == 0:
    raise RuntimeError("No GPU detected. Enable GPU in Kaggle settings.")

strategy = tf.distribute.MirroredStrategy()

print("Execution Mode: PARALLEL")
print("Detected GPUs:", len(gpus))

for gpu in gpus:
    print(gpu.name)

print("Number of replicas:", strategy.num_replicas_in_sync)

training_dir = os.path.join(DATASET_DIR, "Training")
testing_dir = os.path.join(DATASET_DIR, "Testing")

train_paths, train_labels = collect_paths(training_dir)
test_paths, test_labels = collect_paths(testing_dir)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths,
    train_labels,
    test_size=0.20,
    random_state=SEED,
    stratify=train_labels
)

train_ds = make_dataset(train_paths, train_labels, True, "parallel")
val_ds = make_dataset(val_paths, val_labels, False, "parallel")
test_ds = make_dataset(test_paths, test_labels, False, "parallel")

print("\nTraining Images:", len(train_paths))
print("Validation Images:", len(val_paths))
print("Testing Images:", len(test_paths))

with strategy.scope():
    model = build_model()

start = time.perf_counter()

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)

training_seconds = time.perf_counter() - start

start = time.perf_counter()

probabilities = model.predict(test_ds, verbose=0)

prediction_seconds = time.perf_counter() - start

predictions = np.argmax(probabilities, axis=1)

report = classification_report(
    test_labels,
    predictions,
    labels=[0, 1, 2, 3],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)

matrix = confusion_matrix(
    test_labels,
    predictions,
    labels=[0, 1, 2, 3]
)

print("\nPARALLEL CLASSIFICATION REPORT")
print(report)

print("PARALLEL CONFUSION MATRIX")
print(matrix)

total_seconds = training_seconds + prediction_seconds

print(f"\nTraining Time: {training_seconds:.2f} seconds")
print(f"Prediction Time: {prediction_seconds:.2f} seconds")
print(f"Total Time: {total_seconds:.2f} seconds")

parallel_results = {
    "mode": "parallel",
    "number_of_gpus": len(gpus),
    "number_of_replicas": strategy.num_replicas_in_sync,
    "training_seconds": training_seconds,
    "prediction_seconds": prediction_seconds,
    "total_seconds": total_seconds,
    "classification_report": classification_report(
        test_labels,
        predictions,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    ),
    "confusion_matrix": matrix.tolist()
}

model.save("/kaggle/working/brain_tumor_parallel.keras")

with open("/kaggle/working/parallel_results.json", "w") as f:
    json.dump(parallel_results, f, indent=2)

print("\nParallel model and results saved successfully.")

print("\nSERIAL vs PARALLEL")

print(f"Serial Training Time: {serial_results['training_seconds']:.2f} seconds")
print(f"Parallel Training Time: {parallel_results['training_seconds']:.2f} seconds")

print(f"Serial Prediction Time: {serial_results['prediction_seconds']:.2f} seconds")
print(f"Parallel Prediction Time: {parallel_results['prediction_seconds']:.2f} seconds")

print(f"Serial Total Time: {serial_results['total_seconds']:.2f} seconds")
print(f"Parallel Total Time: {parallel_results['total_seconds']:.2f} seconds")

speedup = serial_results["total_seconds"] / parallel_results["total_seconds"]

print(f"\nSpeedup: {speedup:.2f}x")

print("\nSerial Accuracy:",
      serial_results["classification_report"]["accuracy"]*100)

print("Parallel Accuracy:",
      parallel_results["classification_report"]["accuracy"]*100)

print("\nSerial Weighted F1:",
      serial_results["classification_report"]["weighted avg"]["f1-score"])

print("Parallel Weighted F1:",
      parallel_results["classification_report"]["weighted avg"]["f1-score"])


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Execution Mode: PARALLEL
Detected GPUs: 2
/physical_device:GPU:0
/physical_device:GPU:1
Number of replicas: 2

Training Images: 4480
Validation Images: 1120
Testing Images: 1600
Epoch 1/10
INFO:tensorflow:Collective all_reduce tensors: 10 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.6730 - loss: 0.7851 - val_accuracy: 0.8152 - val_loss: 0.4883
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.8132 - loss: 0.4740 - val_accuracy: 0.8643 - val_loss: 0.3866
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.8592 - loss: 0.3928 - val_accuracy: 0.8741 - val_loss: 0.3199
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.8884 - loss: 0.3093 - val_accuracy: 0.8813 - val_loss: 0.3

In [5]:
!git config --global user.name "khushee23"
!git config --global user.email "khuhs2519@gmail.com"

In [6]:
!git clone https://github.com/khushee23/BrainTumorClassification.git

Cloning into 'BrainTumorClassification'...


In [7]:
%cd /kaggle/working/BrainTumorClassification

!git add .
!git commit -m "added code"
!git push

/kaggle/working/BrainTumorClassification
On branch main

Initial commit

nothing to commit (create/copy files and use "git add" to track)
error: src refspec refs/heads/main does not match any
error: failed to push some refs to 'https://github.com/khushee23/BrainTumorClassification.git'
